In [3]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Dataset
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 2. Check Shape, Duplicates, and Data Types
print(f"Dataset Shape: {df.shape}")
print(f"Duplicates: {df.duplicated().sum()}")
print(df.info())

# 3. Data Cleaning
# TotalCharges contains blank spaces ' ' for new customers with tenure = 0
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Binary target flag for numerical operations
df["Churn_Flag"] = df["Churn"].map({"Yes": 1, "No": 0})

# 4. Tenure Binning (0-12, 13-24, 25-48, 49+)
bins = [-1, 12, 24, 48, 100]
labels = ["0-12", "13-24", "25-48", "49+"]
df["Tenure_Group"] = pd.cut(df["tenure"], bins=bins, labels=labels)

print("\nData preparation complete. TotalCharges cleaned and tenure groups created.")

Dataset Shape: (7043, 21)
Duplicates: 0
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBill

In [4]:
# Overall Churn
total_cust = len(df)
churned_cust = df["Churn_Flag"].sum()
retained_cust = total_cust - churned_cust
churn_rate = (churned_cust / total_cust) * 100

print(f"Total Customers: {total_cust:,}")
print(f"Retained Customers: {retained_cust:,}")
print(f"Churned Customers: {churned_cust:,}")
print(f"Overall Churn Rate: {churn_rate:.2f}%\n")

# Contract Churn
contract_churn = df.groupby("Contract")["Churn_Flag"].agg(["count", "mean"])
contract_churn["mean"] = (contract_churn["mean"] * 100).round(2)
print("--- Churn by Contract ---")
print(contract_churn, "\n")

# Tenure Group Churn
tenure_churn = df.groupby("Tenure_Group")["Churn_Flag"].agg(["count", "mean"])
tenure_churn["mean"] = (tenure_churn["mean"] * 100).round(2)
print("--- Churn by Tenure Group ---")
print(tenure_churn, "\n")

# Monthly Charges
print("--- Average Monthly Charges ---")
print(df.groupby("Churn")["MonthlyCharges"].mean().round(2), "\n")

# Payment Method Churn
pm_churn = df.groupby("PaymentMethod")["Churn_Flag"].agg(["count", "mean"])
pm_churn["mean"] = (pm_churn["mean"] * 100).round(2)
print("--- Churn by Payment Method ---")
print(pm_churn, "\n")

# Services: Tech Support & Online Security
print("--- Churn by Tech Support ---")
print((df.groupby("TechSupport")["Churn_Flag"].mean() * 100).round(2), "\n")

print("--- Churn by Online Security ---")
print((df.groupby("OnlineSecurity")["Churn_Flag"].mean() * 100).round(2))

Total Customers: 7,043
Retained Customers: 5,174
Churned Customers: 1,869
Overall Churn Rate: 26.54%

--- Churn by Contract ---
                count   mean
Contract                    
Month-to-month   3875  42.71
One year         1473  11.27
Two year         1695   2.83 

--- Churn by Tenure Group ---
              count   mean
Tenure_Group              
0-12           2186  47.44
13-24          1024  28.71
25-48          1594  20.39
49+            2239   9.51 

--- Average Monthly Charges ---
Churn
No     61.27
Yes    74.44
Name: MonthlyCharges, dtype: float64 

--- Churn by Payment Method ---
                           count   mean
PaymentMethod                          
Bank transfer (automatic)   1544  16.71
Credit card (automatic)     1522  15.24
Electronic check            2365  45.29
Mailed check                1612  19.11 

--- Churn by Tech Support ---
TechSupport
No                     41.64
No internet service     7.40
Yes                    15.17
Name: Churn_Flag, dtype: 

In [5]:
print("================ STATISTICAL HYPOTHESIS TESTS ================")

# Test 1: Contract vs Churn (Chi-Square Test of Independence)
# H0: Contract and Churn are independent
# H1: Contract and Churn are associated
contingency_table = pd.crosstab(df["Contract"], df["Churn"])
chi2, p_chi2, dof, _ = stats.chi2_contingency(contingency_table)

print(f"Test 1 - Chi-Square Test:")
print(f"Chi2 Statistic: {chi2:.4f}, p-value: {p_chi2:.4e}")
print(f"Decision: {'Reject H0' if p_chi2 < 0.05 else 'Fail to Reject H0'}")
print("Interpretation: Contract type and customer churn are significantly associated.\n")

# Test 2: Monthly Charges vs Churn (Independent Samples T-Test)
# H0: Mean monthly charges for churned and retained customers are equal
# H1: Mean monthly charges for churned and retained customers are different
churned_charges = df[df["Churn"] == "Yes"]["MonthlyCharges"]
retained_charges = df[df["Churn"] == "No"]["MonthlyCharges"]
t_stat, p_ttest = stats.ttest_ind(churned_charges, retained_charges, equal_var=False)

print(f"Test 2 - Independent Samples T-Test:")
print(f"Group Mean - Retained: ${retained_charges.mean():.2f} | Churned: ${churned_charges.mean():.2f}")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_ttest:.4e}")
print(f"Decision: {'Reject H0' if p_ttest < 0.05 else 'Fail to Reject H0'}")
print("Interpretation: Churned customers pay significantly higher monthly charges on average.")
print("==============================================================")

================ STATISTICAL HYPOTHESIS TESTS ================
Test 1 - Chi-Square Test:
Chi2 Statistic: 1184.5966, p-value: 5.8630e-258
Decision: Reject H0
Interpretation: Contract type and customer churn are significantly associated.

Test 2 - Independent Samples T-Test:
Group Mean - Retained: $61.27 | Churned: $74.44
t-statistic: 18.4075, p-value: 8.5924e-73
Decision: Reject H0
Interpretation: Churned customers pay significantly higher monthly charges on average.


In [6]:
# Rule-based segments
def assign_segment(row):
    if row["tenure"] <= 12 and row["MonthlyCharges"] >= 65:
        return "High-Risk New"
    elif row["tenure"] <= 12 and row["MonthlyCharges"] < 65:
        return "Standard New"
    elif row["tenure"] > 24 and row["MonthlyCharges"] >= 65:
        return "High-Value Loyal"
    else:
        return "Standard Retained"

df["Customer_Segment"] = df.apply(assign_segment, axis=1)

# Segment evaluation table
segment_metrics = df.groupby("Customer_Segment").agg(
    Customer_Count=("customerID", "count"),
    Avg_Tenure=("tenure", "mean"),
    Avg_Monthly_Charges=("MonthlyCharges", "mean"),
    Churn_Rate=("Churn_Flag", lambda x: round(x.mean() * 100, 2))
)
print("--- Customer Segments ---")
print(segment_metrics.round(2), "\n")

# Revenue at Risk
revenue_at_risk = df[df["Churn"] == "Yes"]["MonthlyCharges"].sum()
print(f"Monthly Revenue at Risk: ${revenue_at_risk:,.2f}")

# Export cleaned data for Power BI
df.to_csv("Telco_Churn_Cleaned.csv", index=False)
print("Exported 'Telco_Churn_Cleaned.csv' for Power BI.")

--- Customer Segments ---
                   Customer_Count  Avg_Tenure  Avg_Monthly_Charges  Churn_Rate
Customer_Segment                                                              
High-Risk New                1001        4.97                81.29       66.43
High-Value Loyal             2379       53.17                92.01       19.46
Standard New                 1185        4.53                34.82       31.39
Standard Retained            2478       36.79                46.24       14.89 

Monthly Revenue at Risk: $139,130.85
Exported 'Telco_Churn_Cleaned.csv' for Power BI.
